# 📘 Week 4 · Day 26-27: 딥러닝 입문 — PyTorch로 Kaggle

## 🎯 학습 목표
- **PyTorch 기본 워크플로우** (Dataset/DataLoader/Module/Train loop) 체화
- **Tabular 데이터 MLP** (Gradient Boosting의 앙상블 파트너)
- **이미지 데이터 CNN** (Digit Recognizer 대회)
- Kaggle GPU 환경에서의 실전 팁

> 🔥 Tabular에선 GBM이 대세지만, **NN을 스태킹에 추가하면** 거의 항상 점수가 오릅니다.
> 이미지/텍스트 대회는 NN이 **필수**.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_classification
from sklearn.metrics import roc_auc_score, accuracy_score
import warnings; warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch {torch.__version__}  device={device}")
torch.manual_seed(42); np.random.seed(42)

---

## 1. PyTorch 핵심 4요소

| 구성 | 역할 |
|------|------|
| **Tensor** | GPU에서 계산 가능한 다차원 배열 |
| **Dataset/DataLoader** | 데이터를 배치로 꺼내는 파이프라인 |
| **nn.Module** | 모델 정의 (layer 조합) |
| **Train Loop** | forward → loss → backward → step |

---

## 2. Tensor 기본

In [ ]:
# 생성
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print("x:\n", x)
print("shape:", x.shape, " dtype:", x.dtype)

# 연산 (numpy와 유사)
print("\nx + 1:\n", x + 1)
print("\nx @ x.T:\n", x @ x.T)

# GPU 이동
x_gpu = x.to(device)
print("\ndevice:", x_gpu.device)

# numpy ↔ tensor
arr = np.array([1, 2, 3])
t = torch.from_numpy(arr)
back = t.numpy()
print("\nnp→t:", t, "  t→np:", back)

### 2.1 Autograd — 자동 미분

In [ ]:
# requires_grad=True → gradient 추적
w = torch.tensor([2.0], requires_grad=True)
b = torch.tensor([1.0], requires_grad=True)
x = torch.tensor([3.0])

y = w * x + b            # forward
loss = (y - 10) ** 2     # target=10

loss.backward()          # backward
print(f"dL/dw = {w.grad.item()}")    # 2(y-10)*x = 2*(-3)*3 = -18
print(f"dL/db = {b.grad.item()}")    # 2(y-10)   = -6

---

## 3. Tabular MLP — GBM의 스태킹 파트너

### 아이디어
Gradient Boosting과 **다른 패턴의 실수**를 해서 앙상블 효과 극대화.

In [ ]:
# 데이터
X_np, y_np = make_classification(n_samples=5000, n_features=20, n_informative=10, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_np, y_np, test_size=0.2, random_state=42, stratify=y_np)

# 스케일링 필수 (NN은 스케일에 민감)
sc = StandardScaler()
X_tr = sc.fit_transform(X_tr)
X_te = sc.transform(X_te)

# Tensor 변환
X_tr_t = torch.FloatTensor(X_tr)
y_tr_t = torch.FloatTensor(y_tr)
X_te_t = torch.FloatTensor(X_te)
y_te_t = torch.FloatTensor(y_te)

# DataLoader
train_ds = TensorDataset(X_tr_t, y_tr_t)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
print(f"train batches: {len(train_loader)}")

### 3.1 MLP 모델 정의

In [ ]:
class TabularMLP(nn.Module):
    def __init__(self, in_dim, hidden=[256, 128, 64], dropout=0.3):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden:
            layers += [
                nn.Linear(prev, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout)
            ]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)

model = TabularMLP(in_dim=X_tr.shape[1]).to(device)
print(model)
print(f"\n총 파라미터: {sum(p.numel() for p in model.parameters()):,}")

### 3.2 학습 루프 (Kaggle 표준 템플릿)

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(xb)
    return total_loss / len(loader.dataset)

@torch.no_grad()
def evaluate(model, X_t, y_t, device):
    model.eval()
    X_t = X_t.to(device)
    logits = model(X_t).cpu().numpy()
    prob = 1 / (1 + np.exp(-logits))
    return roc_auc_score(y_t.numpy(), prob), prob

# 학습
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

best_auc = 0
best_state = None
history = []
patience = 10
bad_epochs = 0

for epoch in range(50):
    tr_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    val_auc, _ = evaluate(model, X_te_t, y_te_t, device)
    scheduler.step(-val_auc)
    history.append({'epoch': epoch, 'loss': tr_loss, 'val_auc': val_auc})

    if val_auc > best_auc:
        best_auc = val_auc
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        bad_epochs = 0
    else:
        bad_epochs += 1
        if bad_epochs >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    if epoch % 5 == 0:
        print(f"epoch {epoch:2d}: loss={tr_loss:.4f}  val_auc={val_auc:.4f}")

model.load_state_dict(best_state)
print(f"\nBest val AUC: {best_auc:.4f}")

In [ ]:
# 학습 곡선
h = pd.DataFrame(history)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(h['epoch'], h['loss']); ax[0].set_title('Train Loss'); ax[0].grid(True)
ax[1].plot(h['epoch'], h['val_auc'], color='green'); ax[1].set_title('Val AUC'); ax[1].grid(True)
plt.tight_layout(); plt.show()

### 3.3 K-Fold NN 템플릿 (스태킹용 OOF 생성)

In [ ]:
from sklearn.model_selection import StratifiedKFold

def train_nn_kfold(X, y, n_splits=5, epochs=30, batch_size=128, lr=1e-3, hidden=[256,128,64]):
    oof = np.zeros(len(y))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    for fold, (tr, val) in enumerate(skf.split(X, y)):
        sc = StandardScaler()
        X_tr_f = sc.fit_transform(X[tr]); X_val_f = sc.transform(X[val])

        ds = TensorDataset(torch.FloatTensor(X_tr_f), torch.FloatTensor(y[tr]))
        loader = DataLoader(ds, batch_size=batch_size, shuffle=True)

        m = TabularMLP(X.shape[1], hidden=hidden).to(device)
        opt = torch.optim.Adam(m.parameters(), lr=lr, weight_decay=1e-5)
        crit = nn.BCEWithLogitsLoss()

        best_state = None; best_auc = 0
        for ep in range(epochs):
            train_epoch(m, loader, opt, crit, device)
            auc, _ = evaluate(m, torch.FloatTensor(X_val_f), torch.FloatTensor(y[val]), device)
            if auc > best_auc:
                best_auc = auc
                best_state = {k: v.clone() for k, v in m.state_dict().items()}

        m.load_state_dict(best_state)
        _, oof[val] = evaluate(m, torch.FloatTensor(X_val_f), torch.FloatTensor(y[val]), device)
        print(f"Fold {fold+1}: AUC={best_auc:.4f}")
    return oof

oof = train_nn_kfold(X_np, y_np, n_splits=5, epochs=25)
print(f"\nOverall OOF AUC: {roc_auc_score(y_np, oof):.4f}")

---

## 4. CNN — Digit Recognizer (MNIST) 대회

### 대회 링크
https://www.kaggle.com/c/digit-recognizer

### 구조
```
Input 28×28×1 → Conv2d → ReLU → MaxPool → ...
                         → Flatten → Linear → Linear(10)
```


In [ ]:
# MNIST 합성 데이터 (실제 대회에선 kaggle에서 다운로드)
# 여기선 모델 구조 시연이 목적
n_sample = 2000
X_img = np.random.rand(n_sample, 1, 28, 28).astype(np.float32)
y_img = np.random.randint(0, 10, n_sample)

# 간단 CNN
class SimpleCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(2)
        self.dropout = nn.Dropout(0.25)
        # 28 → 14 → 7 → 7*7*64 = 3136
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, n_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # (B,32,14,14)
        x = self.pool(F.relu(self.conv2(x)))  # (B,64,7,7)
        x = x.flatten(1)
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x)                    # (B,10) logits

cnn = SimpleCNN().to(device)
print(cnn)
print(f"\n파라미터: {sum(p.numel() for p in cnn.parameters()):,}")

In [ ]:
# CNN 학습 루프 (multi-class)
Xt = torch.FloatTensor(X_img); yt = torch.LongTensor(y_img)
ds = TensorDataset(Xt, yt)
loader = DataLoader(ds, batch_size=64, shuffle=True)

opt = torch.optim.Adam(cnn.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()

for epoch in range(3):
    cnn.train(); total = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = crit(cnn(xb), yb)
        loss.backward(); opt.step()
        total += loss.item() * len(xb)
    print(f"epoch {epoch}: loss={total/len(ds):.4f}")
# 실제 MNIST 데이터면 99%+ 정확도 가능

### 4.1 이미지 대회 실전 팁

1. **Data Augmentation** 필수
   - 회전, crop, flip, mixup, cutmix
2. **Pretrained model** 사용 (전이학습)
   - `torchvision.models.resnet50(pretrained=True)`
   - timm 라이브러리 (`pip install timm`) → SOTA 모델 집합
3. **TTA (Test Time Augmentation)**
   - 예측 시 augment 버전 여러 개 평균
4. **Mixed Precision (AMP)** — 속도 2~3배 향상


In [ ]:
# timm 예시 (설치되어 있다면)
# import timm
# model = timm.create_model('resnet50', pretrained=True, num_classes=10)

# Augmentation 예시
try:
    import torchvision.transforms as T
    transform = T.Compose([
        T.RandomHorizontalFlip(),
        T.RandomRotation(10),
        T.RandomResizedCrop(28, scale=(0.8, 1.0)),
        T.ToTensor(),
        T.Normalize(mean=[0.485], std=[0.229]),
    ])
    print("Augmentation 파이프라인 준비 완료")
except Exception as e:
    print(e)

### 4.2 Mixed Precision Training (AMP)
```python
scaler = torch.cuda.amp.GradScaler()
for xb, yb in loader:
    xb, yb = xb.to(device), yb.to(device)
    opt.zero_grad()
    with torch.cuda.amp.autocast():
        loss = crit(model(xb), yb)
    scaler.scale(loss).backward()
    scaler.step(opt)
    scaler.update()
```

---

## 5. Kaggle GPU 환경 팁

- **Kaggle Notebook 무료 GPU**: T4 ×2 또는 P100, 주당 30시간 (변동)
- GPU 활성화: Notebook 우측 Settings → Accelerator → GPU
- 큰 배치 사이즈 가능 → 속도 ↑
- `torch.cuda.empty_cache()` 로 메모리 관리
- checkpoint 저장: `torch.save(model.state_dict(), '/kaggle/working/model.pt')`

---

## 6. Tabular: NN vs GBM 언제?

| 상황 | NN | GBM |
|------|----|-----|
| 피처 < 30 | GBM 우세 | ✓ |
| 피처 > 100 (sparse) | 비슷 | 비슷 |
| 범주형 고차원 | **Embedding 레이어로 강력** | CatBoost 좋음 |
| 시퀀스/시계열 | **RNN/Transformer 강력** | lag 피처로 경쟁 |
| 데이터 <10K | GBM 우세 | ✓ |
| 데이터 >1M | NN 강력 | 비슷 |

**→ 앙상블에 둘 다 추가가 정답**

---

## 📝 Day 26-27 체크리스트
- [ ] Tensor / Autograd 이해
- [ ] PyTorch Dataset/DataLoader 작성 가능
- [ ] MLP 학습 루프 완전 이해
- [ ] K-Fold NN으로 OOF 생성
- [ ] 간단 CNN 구조 설계

다음은 **텍스트/이미지 고급 + Pseudo-labeling 같은 상위권 기법**입니다.